# 원하는 포즈로 이미지 만드는 도구 (Pose → Image 튜토리얼)

참조 사진 한 장에서 **사람의 관절 위치(포즈)** 만 뽑아내고, 그 포즈를 조건으로 넣어 **같은 자세를 한 전혀 다른 인물·장면**을 만드는 도구입니다.

```
참조 사진 ──(OpenPose)──▶ 관절 뼈대 그림 ──(ControlNet)──┐
                                                        ├──▶ 같은 자세의 새 이미지
프롬프트(누가, 어디서, 어떤 빛, 어떤 스타일) ─────────────┘
```

- **자세**는 참조 사진이 정하고, **인물·배경·화풍**은 프롬프트가 정합니다.
- 위에서부터 셀을 차례로 실행(▶ 또는 Shift+Enter)하면 됩니다. 전체 실행은 `런타임 → 모두 실행`.

## 시작 전 준비 (딱 한 가지)
메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르고 저장합니다. 기본값은 GPU가 없는 쪽이라, 이걸 안 바꾸면 1번 셀에서 바로 멈춥니다.

## 어떤 모델을 쓰나요
| 역할 | 모델 | 라이선스 |
|---|---|---|
| 포즈 추출 | OpenPose (`controlnet_aux`) | – |
| 포즈 조건 회로 | ControlNet v1.1 OpenPose (`lllyasviel/control_v11p_sd15_openpose`) | OpenRAIL |
| 이미지 생성 | Stable Diffusion v1.5 | CreativeML OpenRAIL-M |

> 수업에서는 FLUX.2-klein 4B로 이미지를 만들었지만, FLUX.2에는 아직 공식 ControlNet이 없습니다(교안 6장). 그래서 포즈 ControlNet이 가장 안정적으로 지원되고 무료 T4에서 가볍게 도는 "비슷한 모델"인 Stable Diffusion v1.5로 만들었습니다. 흐름(포즈 추출 → 조건으로 넣기 → 생성)은 어떤 모델에서도 같습니다.

## 1. GPU가 붙었는지 확인
모델은 그래픽카드 메모리(VRAM)에 올라가야 계산이 됩니다. 무엇이 붙었는지부터 확인합니다.

In [ ]:
# [이 셀이 하는 일] GPU가 붙었는지, 이름과 메모리 용량이 얼마인지 확인합니다.
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU가 안 붙었습니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꾸고 다시 실행하세요.")

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} / VRAM {p.total_memory / 1024**3:.1f}GB")

## 2. 필요한 라이브러리 설치
- `diffusers` : 확산 모델을 이름만 대면 내려받아 실행해 주는 도구
- `controlnet_aux` : 사진에서 포즈(관절)를 뽑아 주는 전처리 도구

1분쯤 걸립니다. 빨간 경고 문구가 조금 보여도 마지막에 "설치 완료"가 찍히면 괜찮습니다.

In [ ]:
# [이 셀이 하는 일] 이미지 생성(diffusers)과 포즈 추출(controlnet_aux)에 필요한 라이브러리를 설치합니다.
!pip install -q -U diffusers transformers accelerate controlnet_aux
print("설치 완료")

## 3. 설정값 한곳에 모으기
바꿔 보고 싶은 값은 전부 이 셀에 있습니다. **시드(SEED)를 고정**해 두어야 "결과가 달라진 이유가 내가 바꾼 조건 때문"이라고 말할 수 있습니다.

- 기본값(`USE_MY_PHOTOS = False`)은 이 노트에 **미리 넣어 둔 참조 사진 3장**(한 손을 든 사람 / 의자에 앉은 사람 / 걷는 사람)을 씁니다. 아무것도 올릴 필요 없이 그대로 실행됩니다.
- 다른 사진으로 해 보고 싶으면 `True` 로 바꾸세요. 5번 셀에서 업로드 창이 뜹니다.

In [ ]:
# [이 셀이 하는 일] 시드·이미지 크기·스텝 수 같은 설정값과 결과를 저장할 폴더를 정합니다.
import os, json

SEED = 1234            # 무작위 출발점. 같은 숫자면 같은 조건에서 같은 그림이 나옵니다.
W, H = 512, 768        # 만들 이미지 크기 (가로, 세로). 전신이 들어가도록 세로로 깁니다.
STEPS = 20             # 잡음을 걷어 내는 반복 횟수
GUIDANCE = 7.5         # 프롬프트를 따르는 힘 (이 모델의 일반적인 값)
NEGATIVE = "blurry, low quality, deformed"   # 원하지 않는 것. 짧게만 둡니다.

USE_MY_PHOTOS = False  # False: 노트에 들어 있는 참조 사진 3장 사용. True: 내 사진을 업로드해서 사용.

WORK = "/content/pose-image-tool"
SAMPLES = f"{WORK}/samples"
os.makedirs(SAMPLES, exist_ok=True)
run_log = []           # 어떤 조건으로 무엇을 만들었는지 적어 두는 기록장
print("결과 저장 위치:", SAMPLES)

## 4. 모델 불러오기
세 가지를 불러옵니다. 처음에는 내려받느라 2~4분 걸립니다.

1. **OpenPose** – 사진에서 관절을 찾는 모델
2. **ControlNet** – 관절 그림을 읽어 "이 자세로 그려라"라고 생성 모델에 전달하는 보조 회로
3. **Stable Diffusion v1.5** – 실제로 그림을 그리는 모델

In [ ]:
# [이 셀이 하는 일] 포즈 추출 모델(OpenPose), 포즈 조건 회로(ControlNet), 이미지 생성 모델(SD 1.5)을 내려받아 GPU에 올립니다.
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

# (1) 포즈 추출 도구 불러오기.
#     controlnet_aux 는 다른 전처리 도구까지 한꺼번에 불러오다가 버전 충돌로 실패할 때가 있어서,
#     실패하면 우리가 쓰는 OpenPose 부분만 골라 불러옵니다.
import sys, types
try:
    import scipy.ndimage.filters          # 최신 scipy 에서는 이 옛 경로가 없어졌을 수 있어 대신 채워 둡니다
except Exception:
    import scipy.ndimage
    _m = types.ModuleType("scipy.ndimage.filters")
    _m.gaussian_filter = scipy.ndimage.gaussian_filter
    sys.modules["scipy.ndimage.filters"] = _m

try:
    from controlnet_aux import OpenposeDetector
except Exception as e:
    print("전체 불러오기 실패 → OpenPose 부분만 불러옵니다:", type(e).__name__)
    import importlib.util
    for k in [k for k in sys.modules if k.startswith("controlnet_aux")]:
        del sys.modules[k]
    spec = importlib.util.find_spec("controlnet_aux")
    stub = types.ModuleType("controlnet_aux")
    stub.__path__ = list(spec.submodule_search_locations)
    sys.modules["controlnet_aux"] = stub
    from controlnet_aux.open_pose import OpenposeDetector

openpose = None
for repo in ["lllyasviel/Annotators", "lllyasviel/ControlNet"]:
    try:
        openpose = OpenposeDetector.from_pretrained(repo)
        break
    except Exception as e:
        print(f"{repo} 에서 받기 실패, 다음 후보로:", type(e).__name__)
if openpose is None:
    raise SystemExit("OpenPose 모델을 받지 못했습니다. 잠시 뒤 다시 실행해 보세요.")
try:
    openpose.to("cuda")
except Exception:
    pass  # 안 되면 CPU로 돌립니다 (조금 느릴 뿐 결과는 같습니다)

# (2) 포즈 조건 회로
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_openpose", torch_dtype=torch.float16)

# (3) 이미지 생성 모델 + ControlNet 을 하나의 파이프라인으로 묶기
pipe = None
for repo in ["stable-diffusion-v1-5/stable-diffusion-v1-5", "runwayml/stable-diffusion-v1-5"]:
    try:
        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            repo, controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None)
        BASE_MODEL = repo
        break
    except Exception as e:
        print(f"{repo} 에서 받기 실패, 다음 후보로:", type(e).__name__)
if pipe is None:
    raise SystemExit("이미지 생성 모델을 받지 못했습니다. 잠시 뒤 다시 실행해 보세요.")

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)  # 적은 스텝으로도 잘 나오는 계산 방식
pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

print("모델 준비 완료:", BASE_MODEL)
print(f"사용 중인 VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f}GB")

## 5. 참조 사진 준비 (자세의 출처)
자세를 가져올 사진 3장을 준비합니다.

| 번호 | 참조 사진의 자세 |
|---|---|
| 1 | 서서 **한 손을 머리 옆으로 든** 자세 (정면) |
| 2 | 높은 의자에 **앉아** 신문을 읽는 자세 (비스듬한 각도) |
| 3 | 큰 보폭으로 **걷는** 자세 (옆모습) |

사진 3장은 이 노트 안에 **글자 형태(base64)** 로 들어 있어서 따로 올릴 필요가 없습니다. 아래 셀은 코드가 접혀 있는데, 제목을 더블클릭하면 볼 수 있습니다. 아주 긴 글자 덩어리가 바로 사진입니다.

다른 사진으로 해 보려면 3번 셀에서 `USE_MY_PHOTOS = True` 로 바꾸세요. 업로드 창에서 사진을 1~3장 고르면 파일 이름 순서대로 1·2·3번 포즈가 됩니다. 잘 되는 사진은 ① 사람이 한 명 ② 머리부터 발끝까지 전신 ③ 팔다리가 서로 겹치지 않는 자세입니다.

In [ ]:
#@title 참조 사진 3장 불러오기 (더블클릭하면 코드가 보입니다)
# [이 셀이 하는 일] 자세를 가져올 참조 사진 3장을 준비해 samples/ref_01~03.png 로 저장하고 화면에 보여 줍니다.
import base64, io
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

# 노트에 넣어 둔 참조 사진 (사진 파일을 글자로 바꾼 것)
REF_B64 = {
    "1_hand_up": "...",
    "2_sitting": "...",
    "3_walking": "...",
}

refs = []
if USE_MY_PHOTOS:
    from google.colab import files
    uploaded = files.upload()                      # 업로드 창이 뜹니다
    for name in sorted(uploaded)[:3]:              # 파일 이름 순서대로 1·2·3번 포즈
        img = Image.open(name).convert("RGB")
        img = ImageOps.exif_transpose(img)         # 휴대폰 사진이 옆으로 눕는 것 방지
        refs.append(ImageOps.pad(img, (W, H), color="white"))  # 잘라내지 않고 여백을 넣어 크기 맞춤
else:
    for name in sorted(REF_B64):
        img = Image.open(io.BytesIO(base64.b64decode(REF_B64[name]))).convert("RGB")
        refs.append(img.resize((W, H)))

for i, img in enumerate(refs, start=1):
    img.save(f"{SAMPLES}/ref_{i:02d}.png")
print(f"참조 사진 {len(refs)}장 준비 완료")

## 6. 포즈(관절) 추출
OpenPose가 참조 사진에서 어깨·팔꿈치·손목·골반·무릎·발목 같은 관절을 찾아 **검은 바탕 위의 색깔 막대 그림**으로 바꿉니다. 이 막대 그림이 ControlNet이 읽는 "자세 설명서"입니다. 얼굴 생김새·옷·배경 정보는 여기서 전부 사라지고 자세만 남습니다.

In [ ]:
# [이 셀이 하는 일] 참조 사진마다 관절 뼈대 그림을 뽑아 samples/pose_01~03.png 로 저장하고, 사진과 나란히 보여 줍니다.
import numpy as np

def extract_pose(img):
    # 사진 → 뼈대 그림. 생성할 크기(W×H)에 맞춰 돌려줍니다.
    pose = openpose(img, detect_resolution=512, image_resolution=min(W, H))
    return pose.convert("RGB").resize((W, H))

def pose_found(pose):
    # 뼈대가 하나도 안 그려졌으면(완전히 검은 그림) False
    return int(np.asarray(pose).max()) > 10

def show_row(images, labels, save_as=None):
    # 여러 장을 가로로 이어 붙이고 위에 이름표를 달아 보여 줍니다. (비교용)
    pad = 28
    sheet = Image.new("RGB", (W * len(images), H + pad), "white")
    d = ImageDraw.Draw(sheet)
    for i, (im, lb) in enumerate(zip(images, labels)):
        sheet.paste(im.resize((W, H)), (W * i, pad))
        d.text((W * i + 8, 8), lb, fill="black")
    if save_as:
        sheet.save(f"{SAMPLES}/{save_as}")
    display(sheet.resize((sheet.width // 2, sheet.height // 2)))

poses = []
for i, ref in enumerate(refs, start=1):
    pose = extract_pose(ref)
    if not pose_found(pose):
        print(f"⚠️ {i}번 사진에서 사람을 찾지 못했습니다. 전신이 또렷하게 나온 다른 사진으로 바꿔 보세요.")
    pose.save(f"{SAMPLES}/pose_{i:02d}.png")
    poses.append(pose)
    show_row([ref, pose], [f"ref_{i:02d} (reference photo)", f"pose_{i:02d} (extracted pose)"])

## 7. 생성 함수 만들기
"뼈대 그림 + 프롬프트 → 이미지"를 한 줄로 부를 수 있게 함수로 묶습니다. 만들 때마다 **어떤 조건으로 만들었는지 기록**도 남깁니다. (프롬프트만 적고 시드를 빼먹으면 나중에 재현이 안 됩니다)

In [ ]:
# [이 셀이 하는 일] 포즈 그림과 프롬프트를 받아 이미지를 만들고, 저장하고, 사용한 조건을 기록하는 함수를 정의합니다.
def generate(pose_img, prompt, filename, pose_name, seed=SEED, control_scale=1.0):
    g = torch.Generator(device="cuda").manual_seed(seed)   # 시드 고정
    image = pipe(
        prompt,
        image=pose_img,                               # ← 포즈 조건 (뼈대 그림)
        negative_prompt=NEGATIVE,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=control_scale,  # 포즈를 얼마나 강하게 따를지 (1.0 = 기본)
        width=W, height=H, generator=g,
    ).images[0]
    image.save(f"{SAMPLES}/{filename}")
    run_log.append({"file": filename, "pose": pose_name, "prompt": prompt, "seed": seed,
                    "steps": STEPS, "guidance": GUIDANCE, "control_scale": control_scale,
                    "model": BASE_MODEL, "controlnet": "lllyasviel/control_v11p_sd15_openpose"})
    return image

print("generate() 준비 완료")

## 8. 첫 결과: 참조 사진과 같은 자세로 한 장 만들기
1번 포즈(한 손을 든 자세)에 "공장 작업자" 프롬프트를 넣습니다. 프롬프트에는 **자세에 대한 말이 한마디도 없습니다.** 누가·어디서·어떤 빛·어떤 스타일인지만 적었습니다. 자세는 뼈대 그림이 정합니다.

결과에서 **팔·다리 위치가 참조 사진과 같은지** 확인하세요.

In [ ]:
# [이 셀이 하는 일] 1번 포즈 + 기본 프롬프트로 이미지를 만들어 samples/output_01.png 로 저장하고, 참조 사진·뼈대·결과를 나란히 보여 줍니다.
PROMPT_1 = ("A factory worker in a white safety helmet and navy work uniform, full body, "
            "inside a bright manufacturing plant. Soft morning light from tall windows on the left. "
            "Photorealistic documentary photography, muted industrial color palette.")

out_01 = generate(poses[0], PROMPT_1, "output_01.png", "pose_01")
show_row([refs[0], poses[0], out_01], ["reference", "pose_01", "output_01"], save_as="grid_basic.png")

## 9. 실험 A — 같은 포즈, 다른 프롬프트
포즈(1번)와 시드는 그대로 두고 **프롬프트만** 바꿉니다.

| | 프롬프트 요약 |
|---|---|
| A1 | 공장 작업자, 다큐멘터리 사진 (8번 셀 결과 재사용) |
| A2 | 달 표면의 우주비행사, 영화 스틸 |
| A3 | 주방의 요리사, 수채화 일러스트 |

볼 것: 세 장의 **자세는 같은가?** 인물·배경·화풍은 프롬프트대로 바뀌었는가?

In [ ]:
# [이 셀이 하는 일] 1번 포즈를 고정하고 프롬프트만 바꿔 2장을 더 만든 뒤, 세 결과를 나란히 비교합니다.
PROMPT_2 = ("An astronaut in a white space suit, full body, on the gray surface of the moon "
            "with the Earth in the black sky. Hard sunlight from the right. Photorealistic cinematic still.")
PROMPT_3 = ("A chef in a white uniform and a tall chef hat, full body, in a cozy restaurant kitchen. "
            "Warm light from above. Watercolor illustration, soft pastel colors.")

out_a2 = generate(poses[0], PROMPT_2, "expA_prompt2.png", "pose_01")
out_a3 = generate(poses[0], PROMPT_3, "expA_prompt3.png", "pose_01")
show_row([poses[0], out_01, out_a2, out_a3],
         ["pose_01 (fixed)", "A1 factory worker", "A2 astronaut", "A3 chef / watercolor"],
         save_as="grid_expA_same_pose.png")

## 10. 실험 B — 다른 포즈, 같은 프롬프트
이번에는 반대로 프롬프트(공장 작업자)와 시드는 그대로 두고 **포즈만** 2번(앉은 자세), 3번(걷는 자세)으로 바꿉니다.

볼 것: 같은 "공장 작업자"인데 **자세만 참조를 따라 바뀌는가?** 어느 관절(손·발목 등)이 잘 안 따라오는가?

In [ ]:
# [이 셀이 하는 일] 프롬프트를 고정하고 포즈만 바꿔 output_02, output_03 을 만든 뒤, 포즈와 결과를 위아래 짝으로 비교합니다.
outs_b = [out_01]
for i in range(1, len(poses)):
    outs_b.append(generate(poses[i], PROMPT_1, f"output_{i+1:02d}.png", f"pose_{i+1:02d}"))

show_row(poses, [f"pose_{i+1:02d}" for i in range(len(poses))])
show_row(outs_b, [f"output_{i+1:02d} (same prompt)" for i in range(len(outs_b))],
         save_as="grid_expB_same_prompt.png")

## 11. 실험 C (덤) — 포즈를 따르는 힘 조절
`controlnet_conditioning_scale` 은 "뼈대 그림을 얼마나 강하게 따를지"입니다. 포즈·프롬프트·시드를 모두 고정하고 이 값만 0.3 / 1.0 / 1.6 으로 바꿉니다.

볼 것: 값이 낮을 때 자세가 참조에서 벗어나는가? 값이 높을 때 그림이 부자연스러워지는가?

In [ ]:
# [이 셀이 하는 일] 2번 포즈·같은 프롬프트·같은 시드에서 포즈 강도만 세 단계로 바꿔 비교합니다.
k = min(1, len(poses) - 1)    # 2번 포즈 사용 (사진을 1장만 올렸다면 1번 포즈)
scales = [0.3, 1.0, 1.6]
outs_c = [generate(poses[k], PROMPT_1, f"expC_scale_{s}.png", f"pose_{k+1:02d}", control_scale=s) for s in scales]
show_row([poses[k]] + outs_c, [f"pose_{k+1:02d}"] + [f"scale {s}" for s in scales],
         save_as="grid_expC_scale.png")

## 12. 결과 저장하고 내려받기
지금까지 만든 이미지(`samples/`)와, 사용한 프롬프트·시드 기록(`prompts.md`, `samples/run_log.json`)을 zip 한 개로 묶어 내려받습니다. 압축을 풀어 GitHub 저장소에 올리면 됩니다.

In [ ]:
# [이 셀이 하는 일] 사용한 프롬프트와 조건을 prompts.md / run_log.json 으로 저장하고, samples 폴더와 함께 zip 으로 묶어 내려받습니다.
import shutil

with open(f"{SAMPLES}/run_log.json", "w", encoding="utf-8") as f:
    json.dump(run_log, f, ensure_ascii=False, indent=2)

lines = ["# 테스트에 쓴 프롬프트 모음", "",
         f"- 생성 모델: {BASE_MODEL} + ControlNet v1.1 OpenPose",
         f"- 공통 설정: seed {SEED}, steps {STEPS}, guidance {GUIDANCE}, 크기 {W}x{H}",
         f"- 네거티브 프롬프트: {NEGATIVE}", "",
         "| 결과 파일 | 포즈 | 포즈 강도 | 프롬프트 |", "|---|---|---|---|"]
for r in run_log:
    lines.append(f"| {r['file']} | {r['pose']} | {r['control_scale']} | {r['prompt']} |")
lines += ["", "## 참조 사진", "", "- ref_01.png : 서서 한 손을 머리 옆으로 든 자세", "- ref_02.png : 의자에 앉아 신문을 읽는 자세", "- ref_03.png : 걷는 자세"]
with open(f"{WORK}/prompts.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

zip_path = shutil.make_archive("/content/pose-image-tool_results", "zip", WORK)
print("저장된 파일:", sorted(os.listdir(SAMPLES)))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("왼쪽 파일 창에서 직접 내려받으세요:", zip_path)

## 실험 기록 — 무엇을 바꿔 보았고, 어떻게 달라졌는가

**고정한 것**: 생성 모델(Stable Diffusion v1.5 + ControlNet v1.1 OpenPose), 시드 1234, 스텝 20, 가이던스 7.5, 크기 512×768

**참조 포즈**: pose_01 서서 한 손을 머리 옆으로 든 자세 / pose_02 높은 의자에 앉아 한쪽 무릎을 올린 자세 / pose_03 옆모습으로 크게 걷는 자세

### 실험 A. 같은 포즈 · 다른 프롬프트
- **바꾼 것**: 포즈는 pose_01(서서 한 손을 머리 옆으로 든 자세)로 고정하고, 프롬프트만 ① 공장 작업자·다큐멘터리 사진 → ② 달 위의 우주비행사·영화 스틸 → ③ 주방의 요리사·수채화 일러스트로 바꿨다.
- **자세는 유지됐는가**: 세 장 모두 유지됐다. 한 손은 얼굴 높이까지 들고, 반대쪽 팔은 내리고, 두 발을 모으고 선 전신 자세가 똑같이 나왔다. 화면에서 사람이 서 있는 위치와 크기도 거의 같았다.
- **프롬프트는 반영됐는가**: 인물, 옷, 배경, 화풍이 통째로 바뀌었다. ②는 달 표면과 검은 하늘의 지구가 나왔고, ③은 수채화풍으로 바뀌었다. 재미있는 점은 든 손의 해석이 프롬프트마다 달랐다는 것이다. 참조 사진에서는 구두를 들고 있었는데, 작업자는 공구를, 우주비행사는 손가락을 세워 인사하는 모양을, 요리사는 조리 도구를 들었다. 뼈대에는 "손이 여기 있다"만 있고 무엇을 들었는지는 없어서, 그 빈칸을 프롬프트에 어울리는 것으로 모델이 채운 것이다. 적지 않았는데 생긴 것도 있었다. ①에는 형광 안전조끼가 추가됐고, ③은 오른쪽 아래에 글자처럼 보이는 낙서가 생겼다.

### 실험 B. 다른 포즈 · 같은 프롬프트
- **바꾼 것**: 프롬프트는 "공장 작업자"로 고정하고, 포즈만 pose_01(한 손을 든 자세) → pose_02(의자에 앉은 자세) → pose_03(걷는 자세)로 바꿨다.
- **자세가 참조를 따라왔는가**: 큰 자세는 세 장 모두 따라왔다. pose_02는 앉은 자세, 한쪽 무릎을 올리고 반대쪽 다리를 내린 모양, 몸이 향한 방향이 맞았다. pose_03은 옆모습과 큰 보폭이 그대로 나왔다. 덜 맞은 곳은 팔과 손이었다. pose_02의 참조 사진은 한 손으로 신문을 앞으로 들고 있는데, 결과는 두 팔을 가슴 앞에 모은 모양이 됐다. pose_03은 앞으로 내민 다리가 흰 기계 뒤에 가려졌고, 앞으로 뻗은 손은 그 기계를 붙잡은 모양이 됐다.
- **프롬프트 쪽은 그대로였는가**: 흰 안전모, 남색 작업복, 공장 배경은 세 장 모두 유지됐다. 하지만 같은 프롬프트와 같은 시드인데도 사람은 달라졌다. ①은 수염 난 남자, ②는 여자, ③은 다른 남자였다. 의자는 프롬프트에 없어서 흰 기계 설비로 바뀌었고, 사람이 그 위에 앉아 있다.

### 실험 C. 포즈 강도(controlnet_conditioning_scale) 0.3 / 1.0 / 1.6
- **관찰**: pose_02(앉은 자세)로 실험했다. 0.3에서는 포즈를 거의 따르지 않았다. 앉지 않고 서서 기계 앞에 몸을 숙인 사람이 나왔고, 배경에 작은 사람 두 명이 더 생겼다. 1.0에서는 앉은 자세가 제대로 나왔다. 1.6은 자세는 1.0과 거의 같았지만 그림이 조금 어색해졌다. 안전조끼가 사라지고 소매 색이 중간에 바뀌었으며 손에 천 같은 것이 생겼다. 값을 올린다고 자세가 더 정확해지지는 않았다.

### 정리: 어느 쪽을 바꿨을 때 크게 달라졌고, 어느 쪽이 잘 안 따라왔는가
- 프롬프트를 바꾸면 인물·배경·화풍이 통째로 바뀌고 자세와 구도는 그대로였다. 포즈를 바꾸면 자세와 구도가 바뀌고 옷·배경의 큰 틀은 그대로였다. 두 조건이 맡은 역할이 깔끔하게 나뉘었다.
- 잘 안 따라온 것은 세 가지였다. 첫째, 손과 팔의 세부 모양. 둘째, 같은 인물 유지. 포즈만 바꿔도 사람 얼굴과 성별이 바뀌었다. 셋째, 프롬프트에 없는 소품. 의자나 손에 든 물건은 모델이 알아서 바꾸거나 만들어 냈다.

### 한계
- 뼈대에는 손가락과 손에 든 물건 정보가 없어서 손 모양과 소품은 통제되지 않았다.
- 포즈 조건은 자세만 잡아 줄 뿐 같은 인물을 유지해 주지 않는다. 같은 사람을 여러 자세로 만들려면 다른 방법이 더 필요하다.
- 포즈 강도가 낮으면(0.3) 조건이 사실상 무시되고, 높으면(1.6) 그림이 어색해진다. 1.0 근처가 무난했다.
- 화면 안 글자는 제대로 나오지 않았다(수채화 결과의 낙서).